# 모델의 발목점은 어느 기준점에 더 가까울까?

무릎을 통과하는 같은 세 프레임에서 **모델점부터 복사뼈까지의 거리**와 **모델점부터 앞쪽 접힘점까지의 거리**를 비교합니다. 접힘점은 트레이너가 정한 **정강이 앞쪽 윤곽이 발등으로 꺾이는 한 점**입니다.

**Run All**을 누르고 각 탭에서 가까운 다리의 접힘점 하나를 클릭하세요. 기존 복사뼈와 모델점은 판독 화면에서 숨깁니다. 잘 보이지 않으면 **원본 크기로 확대**를 켜세요. 윤곽을 짐작해 찍었다면 **가림 추정점으로 변경**, 위치를 정하기 어려우면 **위치 불확실** 또는 **가려서 판독 불가**를 누르세요. 신발 입구나 끈만 보인다면 그 선을 접힘점으로 대신 찍을 필요는 없습니다.

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

from wlcv.annotation.pose_points import ANKLE_FOLD, load_points, pose_point_editor, pose_point_path
from wlcv.config import ANNOTATIONS, ROOT, SPLIT, VIDEOS
from wlcv.data.frames import read_rgb
from wlcv.evaluation.ankle import compare_ankle_landmarks

# 19번 실험과 같은 선수·프레임을 사용합니다.
frames = {"a05_sn_f_01": 18, "b03_sn_f_01": 24, "c08_sn_f_01": 24}
split = pd.read_csv(SPLIT)
validation = split.loc[split["role"] == "validation", ["video_id", "person_id"]]
metadata = pd.read_csv(ROOT / "data/metadata/videos.csv")
videos = validation.merge(metadata[["video_id", "filename", "near_side"]], on="video_id",
                          validate="one_to_one").set_index("video_id").loc[list(frames)]
output_dir = ROOT / "outputs/pose_baseline"

In [ ]:
editors = []
for video, frame in frames.items():
    info = videos.loc[video]
    editor = pose_point_editor(VIDEOS / info.filename, frame, info.person_id, info.near_side,
                               split="validation", joints=(ANKLE_FOLD,))
    editors.append(editor)

tabs = widgets.Tab(children=editors)
for n, (video, frame) in enumerate(frames.items()):
    tabs.set_title(n, f"{video} · {frame}번")
display(tabs)

## 세 장을 판독한 뒤 마지막 셀만 다시 실행하세요

19번 실험에 저장된 복사뼈와 모델 좌표를 그대로 사용합니다. 그림의 분홍 십자는 복사뼈, 노란 삼각형은 접힘점, 청록 원은 모델점입니다. 표는 같은 프레임 안에서 원본 픽셀 단위의 직선거리를 비교합니다.

**거리 차이 = 모델–복사뼈 거리 − 모델–접힘점 거리**입니다. 양수이면 접힘점, 음수이면 복사뼈에 더 가깝습니다. 0이면 거리가 같습니다. 추정으로 표시한 좌표는 그림과 참고 거리로만 남기고, 판독 불가인 점의 거리는 비워 둡니다. 아주 작은 차이는 클릭 위치가 조금만 달라져도 순서가 바뀔 수 있습니다.

In [ ]:
folds = {}
for video, frame in frames.items():
    path = pose_point_path(ANNOTATIONS, video, frame, fold=True)
    point = load_points(path).get(ANKLE_FOLD)
    if point is not None:
        info = videos.loc[video]
        if (point["video_id"] != video or int(point["frame_index"]) != frame
                or point["person_id"] != info.person_id or point["side"] != info.near_side
                or point["split"] != "validation"):
            raise ValueError(f"{video}의 판독과 선수·프레임 정보를 확인하세요.")
        folds[video] = point

pending = [video for video in frames if video not in folds]
if pending:
    print("접힘점을 판독할 영상:", ", ".join(pending))
else:
    previous = pd.read_csv(output_dir / "transition_ankles.csv")
    previous = previous.set_index(["영상", "프레임", "가까운 쪽"], verify_integrity=True)
    rows = []
    for video, frame in tqdm(frames.items(), desc="두 기준점 비교", unit="장"):
        info = videos.loc[video]
        bone_and_model = previous.loc[(video, frame, info.near_side)]
        rgb = read_rgb(VIDEOS / info.filename, frame)
        row, fig = compare_ankle_landmarks(rgb, bone_and_model, folds[video])
        rows.append(row)
        plt.show()

    table = pd.DataFrame(rows)
    display(table.style.format(precision=2, na_rep="—"))
    direct = table["더 가까운 기준점"].dropna()
    print(f"직접 비교 가능 {len(direct)}/{len(table)}장 · 접힘점 {(direct == '접힘점').sum()}장 · "
          f"복사뼈 {(direct == '복사뼈').sum()}장 · 같은 거리 {(direct == '같은 거리').sum()}장")
    table.to_csv(output_dir / "transition_ankle_fold.csv", index=False)